In [1]:
import os
os.environ["UNSLOTH_SKIP_TORCHVISION_CHECK"] = "1"
import torch
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments

# 1. 环境与路径配置
MODEL_NAME = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"
TRAIN_FILE = "../yue/yue_train.jsonl"
VAL_FILE = "../yue/yue_val.jsonl"
OUTPUT_DIR = "outputs_yue_qwen"

# 2. 加载模型与分词器
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_NAME,
    max_seq_length = 2048,
    load_in_4bit = True,
)

# 3. 配置 LoRA 适配器
# 这里使用了 LoRA 秩 $r=16$ 和 缩放系数 $\alpha=32$
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, 
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

# 4. 准备数据集
dataset = load_dataset("json", data_files={"train": TRAIN_FILE, "eval": VAL_FILE})

# 构造 Qwen ChatML 格式的 Prompt
def formatting_prompts_func(examples):
    inputs = examples["input"]
    outputs = examples["output"]
    texts = []
    for i, o in zip(inputs, outputs):
        text = f"<|im_start|>system\n你是一个地道的粤语翻译助手。<|im_end|>\n<|im_start|>user\n{i}<|im_end|>\n<|im_start|>assistant\n{o}<|im_end|>"
        texts.append(text)
    return { "text" : texts }

dataset = dataset.map(formatting_prompts_func, batched = True, num_proc = 4)

# 5. 配置训练参数
# 对于 213 万条数据，我们不按 Epoch 跑，而是按 Step 跑，先定 10,000 步
# 5. 配置训练参数
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset["train"],
    eval_dataset = dataset["eval"],
    dataset_text_field = "text",
    max_seq_length = 2048,
    dataset_num_proc = 4,
    packing = True, 
    args = TrainingArguments(
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 4,
        warmup_steps = 200,
        max_steps = 10000,
        learning_rate = 1e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = OUTPUT_DIR,
        eval_strategy = "steps",  # 👈 这里改成了 eval_strategy
        eval_steps = 1000,
        save_steps = 1000,
        save_total_limit = 2,
        report_to = "none",
    ),
)

# 6. 开始训练
print("🔥 粤语翻译模型微调正式启动...")
trainer_stats = trainer.train(resume_from_checkpoint = True)

# 7. 保存 LoRA 权重
model.save_pretrained("yue_qwen_lora")
tokenizer.save_pretrained("yue_qwen_lora")
print("✅ 训练完成，模型已保存至 yue_qwen_lora")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/usr/local/miniconda3/envs/py310/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/usr/local/miniconda3/envs/py310/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'Could not load this library: /usr/local/miniconda3/envs/py310/lib/python3.10/site-packages/torchvision/image.so'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(


Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.6: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 3090. Num GPUs = 1. Max memory: 23.558 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|██████████| 339/339 [00:01<00:00, 291.30it/s]


unsloth/Qwen2.5-7B-Instruct-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


...

🔥 粤语翻译模型微调正式启动...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,130,950 | Num Epochs = 1 | Total steps = 10,000
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 40,370,176 of 7,655,986,688 (0.53% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss,Validation Loss
9000,1.387751,1.392519
10000,1.445427,1.390062


/usr/local/miniconda3/envs/py310/lib/python3.10/site-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/miniconda3/envs/py310/lib/python3.10/site-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/miniconda3/envs/py310/lib/python3.10/site-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated 

✅ 训练完成，模型已保存至 yue_qwen_lora
